<a href="https://colab.research.google.com/github/nukegara64/enneagram-personality-lora-trainer/blob/main/Enneagram_Personality_LoRA_Trainer_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ==========================================================
# Enneagram Personality LoRA Trainer
# ==========================================================

 用途:
 エニアグラム各タイプの「人格・価値観・口調」を
 個別LoRAとして学習するためのプログラム。

 タイプごとに独立したLoRAを作成し、
 推論時に切り替えることで人格を再現する。

 目的:
 特性一覧を暗唱するAIではなく、
 その人格らしい判断・発言を行うAIを作ること。

 注意:
 タイプ1〜9のデータは混ぜない。
 1タイプ = 1LoRA。

 成功判定:
 未知の質問に対しても
 そのタイプらしい回答が自然に出ること。

#Cell　0. Colabを開く

ランタイムをGPUにします。

ランタイム → ランタイムのタイプを変更 → T4 GPU


Type2以降

ランタイムを再起動してから、Cell 2だけ変更します。

#Cell 1：インストール

In [ ]:
!pip install -U unsloth trl datasets accelerate bitsandbytes peft transformers

#Cell 2：設定

LoRAの保存名は、OUTPUT_DIR = "type1_lawman_corrector_lora"です
タイプごとに変更してください

1.   Type1 正論タイラント
2.   Type2 恩着せパラディン
3.   Type3 成果アサシン
4.   Type4 虚無ネクロマンサー
5.   Type5 知識スナイパー
6.   Type6 忠誠タンク
7.   Type7 自由レンジャー
8.   Type8 威圧バーサーカー
9.   Type9 空気バッファー

 各タイプは別LoRAで学習すること
 結合学習禁止
 人格汚染防止

In [ ]:
DATA_FILE = "/content/type1_lawman_corrector_lora_100_fixed.jsonl"
OUTPUT_DIR = "type1_lawman_corrector_lora"
BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024

In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return texts

#Cell 3：JSONLをアップロード

In [ ]:
from google.colab import files
uploaded = files.upload()

#Cell 4：モデル読み込み

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

#Cell 5：データ読み込み

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files = DATA_FILE,
    split = "train",
)

print(dataset[0])

#Cell 6：学習用フォーマット関数

In [ ]:
def formatting_prompts_func(examples):
    texts = []

    for messages in examples["messages"]:
        system_text = ""
        user_text = ""
        assistant_text = ""

        # messages が {"role":[...], "content":[...]} 形式の場合
        if isinstance(messages, dict):
            roles = messages.get("role", [])
            contents = messages.get("content", [])

            for role, content in zip(roles, contents):
                if role == "system":
                    system_text = content
                elif role == "user":
                    user_text = content
                elif role == "assistant":
                    assistant_text = content

        # messages が [{"role":"...", "content":"..."}] 形式の場合
        elif isinstance(messages, list):
            for m in messages:
                if not isinstance(m, dict):
                    continue

                role = m.get("role", "")
                content = m.get("content", "")

                if role == "system":
                    system_text = content
                elif role == "user":
                    user_text = content
                elif role == "assistant":
                    assistant_text = content

        text = f"""### System:
{system_text}

### User:
{user_text}

### Assistant:
{assistant_text}"""

        texts.append(text)

    return texts

#Cell 7：学習



 #人格LoRA チューニング指針


 #データ品質 >>> データ量 >> max_steps > r > learning_rate

 #■ 人格が薄い
 #max_steps += 40
 #r = 16 → 32

 #■ セリフを丸暗記する
 #max_steps -= 30
 #r = 16 → 8

 #■ 学習が不安定
 #learning_rate = 2e-4 → 1e-4

 #■ 最も効果が大きい改善
 #データ100件 → 300件以上

 #■ 悪い学習データ
 #「高潔さ」
 #「知識欲」
 #「自由」

 #■ 良い学習データ
 #「規律は窮屈だ。だが崩壊よりはましだ。」
 #「知らないまま決断するな。」
 #「逃げ道があるなら全部試してから絶望しろ。」

 #■ 推奨設定（人格LoRA）
 #r = 8～16
 #learning_rate = 2e-4
 #max_steps = 50～120

 #■ 判定方法
 #Lossではなく未知の質問への返答を見る

 #同じ質問で
 #Type1 → 規律・責任
 #Type5 → 知識・観察
 #Type8 → 力・支配
 #Type9 → 調和・回避

 #が自然に出れば成功


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    formatting_func = formatting_prompts_func,
    args = SFTConfig(
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 80,
        learning_rate = 2e-4,
        logging_steps = 5,
        output_dir = OUTPUT_DIR,
        packing = False,
    ),
)

trainer.train()

#Cell 8：LoRA保存

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

#Cell 9：Google Driveに保存

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!cp -r "{OUTPUT_DIR}" "/content/drive/MyDrive/{OUTPUT_DIR}"

# Cell 10：確認方法

adapter_model.safetensors　LoRA本体は実質これです

adapter_config.json　　　　これが人格データ

In [ ]:
!ls type1_lawman_corrector_lora